In [2]:
import pandas as pd
import re

In [3]:
# Read the csv as a pandas dataframe
df = pd.read_csv("../data/messy_samples.csv")

In [4]:
# Clean the sample_id column by converting to uppercase and removing hyphens
df["sample_id"] = df["sample_id"].str.upper().str.replace(
    r"-", "", regex=True
)

In [5]:
# Replace periods with slashes, convert to datetime, and format as YYYY-MM-DD
df["dob"] = pd.to_datetime(
    df["dob"].str.replace(r"\.", "/", regex=True),
    format="mixed",
    errors="coerce"
).dt.strftime("%Y-%m-%d")

In [6]:
# Convert sex column to uppercase and replace values with standardized ones
df["sex"] = df["sex"].str.upper().str.replace(
    r"MALE", "M", regex=True
).str.replace(
    r"FEMALE", "F", regex=True
).str.replace(
    r"U|UNKNOWN", "Unknown", regex=True
)

In [7]:
# Strip whitespace, convert to uppercase, and remove spaces, underscores, and hyphens
df["enrollment_site"] = (
    df["enrollment_site"]
    .str.strip()
    .str.upper()
    .str.replace(r"[\s_-]+", "", regex=True)
)

# Replace enrollment site codes with standardized names
df["enrollment_site"] = df["enrollment_site"].str.replace(
    r"SITEA", "Site A", regex=True
).str.replace(
    r"SITEB", "Site B", regex=True
).str.replace(
    r"SITEC", "Site C", regex=True
)

In [8]:
# Remove asterisks and convert to numeric
df["glucose_value"] = pd.to_numeric(
    df["glucose_value"].astype(str).str.replace(r"\*", "", regex=True),
    errors="coerce"
)

# Convert glucose_unit to lowercase
df["glucose_unit"] = df["glucose_unit"].str.lower()

# Convert mmol/L → mg/dL
mmol = df["glucose_unit"] == "mmol/l"

# Convert mmol/L to mg/dL using the conversion factor 18.0182
df.loc[mmol, "glucose_value"] = (
    df.loc[mmol, "glucose_value"] * 18.0182
).round(1)

# Set glucose_unit to mg/dL for all rows
df["glucose_unit"] = "mg/dL"

In [9]:
# Save output
df.to_csv("../data/cleaned_samples_regex.csv", index=False)

In [18]:
# Graduate Addendum

df = pd.read_csv("../data/cleaned_samples_regex.csv")

feature_table = df[
    ["sample_id", "glucose_value", "dob", "sex", "enrollment_site"]
]

print(feature_table.dtypes)

print("\nMissing values:")
print(feature_table.isna().sum())

sample_id              str
glucose_value      float64
dob                    str
sex                    str
enrollment_site        str
dtype: object

Missing values:
sample_id          0
glucose_value      0
dob                0
sex                5
enrollment_site    0
dtype: int64
